# 2 · Predicting Novelty with the Novelty Reviewer Model

[`AbeerMostafa/Novelty_Reviewer`](https://huggingface.co/AbeerMostafa/Novelty_Reviewer) is Llama-3.1-8B-Instruct
fine-tuned on the dataset. Given a paper, it returns a novelty score in {-1, 0, 1, 2} and a reviewer-style assessment.

**Requirements:** a GPU with ≥ 24 GB memory (bf16) and access to the Llama 3.1 license on Hugging Face
(`huggingface-cli login`).

In [ ]:
# Install the package (run once). In Colab, clone the repository first:
# !git clone https://github.com/m2l-lab/Novelty-Reviewer && cd Novelty-Reviewer
# !pip install -q -e "Novelty-Reviewer[model]"

In [ ]:
from novelty_reviewer import (NOVELTY_LABELS, NoveltyReviewer, download_openreview_pdf, eval_subset,
                              load_novelty_dataset)
from novelty_reviewer.pdf import pdf_to_markdown

reviewer = NoveltyReviewer.from_pretrained()  # downloads ~16 GB on first use

## Getting a paper

The model expects the full paper as markdown. We take a submission from the evaluation subset and download it from
OpenReview. If OpenReview blocks anonymous downloads from your network, set `OPENREVIEW_USERNAME` and
`OPENREVIEW_PASSWORD`, or download the PDF manually.

In [ ]:
papers = eval_subset(load_novelty_dataset("test"))
paper = papers[0]
print(paper["title"])

pdf_path = download_openreview_pdf(paper["paper_id"], "papers")
paper_text = pdf_to_markdown(pdf_path)
print(f"{len(paper_text):,} characters")

## Predicting

In [ ]:
prediction = reviewer.predict(paper_text)
print(prediction)

In [ ]:
print("Predicted:", prediction.score, f"({prediction.label})")
print("Reviewers (ECNA):", paper["novelty_score"], f"({NOVELTY_LABELS[paper['novelty_score']]})\n")
print(paper["novelty_score_justification"])

## Your own manuscript

Any PDF works the same way:

```python
prediction = reviewer.predict(pdf_to_markdown("my_paper.pdf"))
```

or from the command line:

```bash
novelty-reviewer predict --pdf my_paper.pdf
novelty-reviewer predict --paper-id <openreview-id>
```

## Under the hood

`NoveltyReviewer` wraps the exact prompt used for fine-tuning; `build_messages` shows it. Generation defaults to the
decoding of the paper (temperature 0.1, top-p 0.5, 300 new tokens) and can be overridden in `predict`.

In [ ]:
print(NoveltyReviewer.build_messages("<paper text>")[0]["content"])
greedy = reviewer.predict(paper_text, do_sample=False, max_new_tokens=400)